### Data
We use the course data in the folder `Data` (place it next to this notebook):
* `GPL1261-56135.tsv` – annotation of the Affymetrix Mouse Genome 430 2.0 array (45,101 probes, 64 MB on disk).
* `primates_diet.tsv` – expression of the 43,014 probes in 24 mouse samples: four diets (Chimpanzee, FastFood, HumanCafe, Pellet) × 2 batches × 3 replicates.

These files are small enough to experiment with in class, but large enough to *see* the effects of the
techniques. Everything we do scales unchanged to files that are many times larger than your RAM.

In [ ]:
import csv
import heapq
import sqlite3
import time
import tracemalloc
from collections import Counter
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

# --- locate the course data folder -------------------------------------------
CANDIDATES = [Path("Data"), Path("../Data"), Path("../../Data")]
DATA_DIR = next((p for p in CANDIDATES if (p / "primates_diet.tsv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Put the course folder 'Data' (with GPL1261-56135.tsv and "
                            "primates_diet.tsv) next to this notebook.")

ANNOTATION = DATA_DIR / "GPL1261-56135.tsv"
EXPRESSION = DATA_DIR / "primates_diet.tsv"
LONG_FILE = Path("primates_diet_long.tsv")    # created in Part 2
DB_FILE = Path("primates_diet.sqlite")         # created in Part 3


@contextmanager
def measure(label):
    # Print wall-clock time and peak Python memory of the indented block.
    tracemalloc.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print(f"{label:<42} {elapsed:6.2f} s   peak memory {peak / 1e6:7.1f} MB")


def mb(df):
    # Memory of a DataFrame/Series in MB, including the content of strings.
    return df.memory_usage(deep=True).sum() / 1e6 if isinstance(df, pd.DataFrame) \
        else df.memory_usage(deep=True) / 1e6


for f in (ANNOTATION, EXPRESSION):
    print(f"{f.name:<25} {f.stat().st_size / 1e6:6.1f} MB on disk")

<details><summary><b>Automatic checks for the exercises</b> – run the next cell once, you do not need to read it.</summary>
The functions <code>check_exercise_1()</code>, <code>check_exercise_2()</code> and <code>check_exercise_3()</code> compare
your results with a reference computed in memory with pandas.
</details>

In [ ]:
def _report(task, func):
    # Run one check; print a tick, a cross or 'not solved yet'.
    try:
        ok, detail = func()
        print(("✔ " if ok else "✘ ") + task + (f" – {detail}" if detail else ""))
    except NotImplementedError:
        print(f"… {task} – not solved yet")
    except Exception as err:                                   # noqa: BLE001
        print(f"✘ {task} – {type(err).__name__}: {err}")


def _reference_expression():
    expr = pd.read_csv(EXPRESSION, sep="\t")
    values = expr.set_index("ID_REF")
    chimp = values.filter(like="Chimpanzee").mean(axis=1)
    pellet = values.filter(like="Pellet").mean(axis=1)
    return expr, values, (chimp / pellet).sort_values(ascending=False)


def check_exercise_1(stream_expression, sample_means, count_high_probes, top_chimp_vs_pellet):
    expr, values, ratio = _reference_expression()

    def a():
        items = stream_expression(EXPRESSION)
        first = next(iter(items))
        expected = ("1415670_at", [57, 73, 83, 33, 24, 37, 60, 65, 59, 40, 24, 39,
                                   49, 61, 73, 39, 38, 23, 97, 72, 88, 58, 62, 71])
        n = 1 + sum(1 for _ in items)
        ok = first == expected and n == len(expr) and not isinstance(items, list)
        return ok, f"{n} records, first = {first[0]}" + ("" if ok else " (expected a lazy generator of 43014 tuples)")

    def b():
        means = sample_means(EXPRESSION)
        ok = all(abs(means[s] - values[s].mean()) < 1e-6 for s in values.columns)
        return ok, f"mean of {values.columns[0]} = {means[values.columns[0]]:.2f}"

    def c():
        n = count_high_probes(EXPRESSION, threshold=1000)
        expected = int((values.max(axis=1) > 1000).sum())
        return n == expected, f"{n} probes (expected {expected})"

    def d():
        top = top_chimp_vs_pellet(EXPRESSION, k=5)
        ok = [p for _, p in top] == list(ratio.index[:5])
        return ok, ", ".join(f"{p}: {r:.2f}" for r, p in top)

    for name, f in [("1A stream_expression", a), ("1B sample_means", b),
                    ("1C count_high_probes", c), ("1D top_chimp_vs_pellet (bonus)", d)]:
        _report(name, f)


def _reference_annotation():
    ann = pd.read_csv(ANNOTATION, sep="\t", usecols=["ID", "Gene Symbol", "Sequence Type"])
    ann = ann[(ann["Sequence Type"] != "Control sequence") & ann["Gene Symbol"].notna()]
    return pd.DataFrame({"probe_id": ann["ID"].to_numpy(),
                         "symbol": ann["Gene Symbol"].str.split(" /// ").str[0].to_numpy()})


def _reference_diet_means():
    expr = pd.read_csv(EXPRESSION, sep="\t").melt(id_vars="ID_REF", var_name="sample", value_name="value")
    expr["diet"] = expr["sample"].str.extract(r"^(\w+?)Diet", expand=False)
    merged = expr.merge(_reference_annotation(), left_on="ID_REF", right_on="probe_id")
    return merged.groupby(["symbol", "diet"])["value"].mean().unstack("diet")


def check_exercise_2(load_expression_small, load_annotation_small, diet_means_chunked, top_fastfood_vs_pellet):
    default_mb = mb(pd.read_csv(EXPRESSION, sep="\t"))
    state = {}

    def a():
        small = load_expression_small(EXPRESSION)
        ratio = default_mb / mb(small)
        ok = ratio >= 2 and int(small.select_dtypes("number").to_numpy(dtype="int64").sum()) == \
            int(pd.read_csv(EXPRESSION, sep="\t").iloc[:, 1:].to_numpy().sum())
        return ok, f"{default_mb:.1f} MB -> {mb(small):.1f} MB ({ratio:.1f}x smaller, goal: at least 2x)"

    def b():
        ann = load_annotation_small(ANNOTATION)
        ref = _reference_annotation()
        ok = list(ann.columns) == ["probe_id", "symbol"] and len(ann) == len(ref) \
            and isinstance(ann["symbol"].dtype, pd.CategoricalDtype)
        state["annotation"] = ann
        return ok, f"{len(ann)} probes, {mb(ann):.1f} MB, symbol dtype = {ann['symbol'].dtype}"

    def c():
        ann = state.get("annotation", _reference_annotation())
        means = diet_means_chunked(LONG_FILE, ann, chunksize=100_000)
        ref = _reference_diet_means()
        means = means.loc[ref.index, ref.columns]
        state["means"] = means
        return bool(np.allclose(means.to_numpy(), ref.to_numpy())), f"{means.shape[0]} genes x {means.shape[1]} diets"

    def d():
        means = state.get("means", _reference_diet_means())
        top = top_fastfood_vs_pellet(means, k=10)
        ref = np.log2((means["FastFood"] + 1) / (means["Pellet"] + 1)).sort_values(ascending=False).head(10)
        ok = list(top.index) == list(ref.index) and np.allclose(top.to_numpy(), ref.to_numpy())
        return ok, ", ".join(f"{g} ({v:.2f})" for g, v in top.head(3).items()) + ", ..."

    for name, f in [("2A load_expression_small", a), ("2B load_annotation_small", b),
                    ("2C diet_means_chunked", c), ("2D top_fastfood_vs_pellet", d)]:
        _report(name, f)


def check_exercise_3(query):
    def a():
        if query is None:
            raise NotImplementedError
        with sqlite3.connect(DB_FILE) as con:
            result = pd.read_sql_query(query, con)
            ref = pd.read_sql_query(
                "SELECT a.symbol, AVG(e.value) AS mean_value FROM expression e "
                "JOIN annotation a ON e.probe_id = a.probe_id WHERE e.diet = 'HumanCafe' "
                "GROUP BY a.symbol ORDER BY mean_value DESC LIMIT 5", con)
        ok = list(result.iloc[:, 0]) == list(ref["symbol"]) and np.allclose(result.iloc[:, 1], ref["mean_value"])
        return ok, ", ".join(result.iloc[:, 0].astype(str))

    _report("3A SQL query", a)

---
# Streaming in plain Python

## 1.1 Reading everything vs. streaming
A file object is an **iterator over lines**. `f.readlines()` builds a list with *all* lines in memory;
`for line in f` keeps only *one* line at a time. Both visit every line once (O(n) time),
but the memory need is O(n) vs. O(1).

## 1.2 Generators: functions that *yield*
A function containing `yield` returns a **generator**. Its body runs only when someone asks for the next
value, and it pauses after each `yield`. You already know this from `LinkedList.__iter__` in Session 8.

We use the `csv` module because fields of the annotation file can contain quotes and separators.

## 1.3 Generator pipelines
Small generators can be chained like a pipeline: *read → filter → transform → aggregate*.
Each record flows through the whole pipeline before the next one is read.

## 1.4 Streaming statistics
Many statistics can be **updated record by record**: count, sum, mean, minimum/maximum, variance
(via the sum of squares) and even the *k* largest values (`heapq.nlargest` keeps only *k* items).
The median cannot – it needs all values (or a sorted file).

---
# Exercise 1 – Streaming the expression file
Work **only with streaming**: never build a list of all lines or records.
The helper `read_header` is given.

* **1A** Write the generator `stream_expression(path)` that yields one tuple `(probe_id, values)` per data line
  of `primates_diet.tsv`, where `values` is a list of 24 integers.
* **1B** Write `sample_means(path)` that returns a dictionary *sample name → mean expression* using
  running sums over `stream_expression`.
* **1C** Write `count_high_probes(path, threshold)` that counts the probes whose maximum over all samples is
  larger than `threshold`.
* **1D (bonus)** Write `top_chimp_vs_pellet(path, k)` that returns the *k* probes with the largest ratio
  *mean(Chimpanzee samples) / mean(Pellet samples)* as a list of `(ratio, probe_id)`, largest first.
  Use `heapq.nlargest` on a generator expression.
* **1E (discussion)** Use `measure(...)` to compare the peak memory of your `sample_means` with a version that
  first calls `f.readlines()`. What changes if the file were 100× larger?

In [ ]:
def read_header(path):
    # Return the list of column names of a tab-separated file.
    with open(path, encoding="utf-8") as f:
        return f.readline().rstrip("\n").split("\t")


print(read_header(EXPRESSION)[:4], "...")

In [ ]:
def stream_expression(path):
    # TODO 1A: open the file, skip the header, yield (probe_id, [24 ints]) per line
    raise NotImplementedError


def sample_means(path):
    # TODO 1B: running sums per sample -> dict {sample: mean}
    raise NotImplementedError


def count_high_probes(path, threshold=1000):
    # TODO 1C
    raise NotImplementedError


def top_chimp_vs_pellet(path, k=5):
    # TODO 1D (bonus): heapq.nlargest(k, <generator expression of (ratio, probe_id)>)
    raise NotImplementedError

In [ ]:
check_exercise_1(stream_expression, sample_means, count_high_probes, top_chimp_vs_pellet)

In [ ]:
# TODO 1E: compare the peak memory, e.g.
# with measure("streaming sample_means"):
#     sample_means(EXPRESSION)
# with measure("readlines() version"):
#     ...

# pandas for large data

## 2.1 Load only what you need: `usecols`, `nrows`, `dtype`
By default pandas stores text as Python objects (`object`) and numbers as 64-bit types.
`memory_usage(deep=True)` shows the real memory, including the content of strings.

## 2.2 Choose small dtypes: downcasting and `category`
* Integers from 4 to 8255 fit into `int16` (–32768 … 32767): 2 bytes instead of 8.
* Columns with few distinct text values (diet, species, chromosome) become `category`:
  every value is stored once, the rows only keep a small integer code.

## 2.3 Writing in chunks: from wide to long format
Large omics tables are often stored in **long format** (one measurement per row). We create such a file
from the expression table – chunk by chunk, so that only a few thousand probes are in memory at a time.
`mode="a"` appends to the file; only the first chunk writes the header.

## 2.4 Reading in chunks: `chunksize`
With `chunksize`, `read_csv` returns an **iterator of DataFrames** – pandas' version of streaming.
Aggregate every chunk, keep only the small partial results, and combine them at the end.

⚠️ **Combine sums and counts, not means.** The mean of chunk means is wrong when the groups have
different sizes in different chunks.

# Exercise 2 – An out-of-core pandas pipeline
**Question:** which genes react most strongly to the fast-food diet compared to the pellet (standard) diet?
Pretend that the long file `primates_diet_long.tsv` is too large to load at once.

* **2A** `load_expression_small(path)` – load `primates_diet.tsv` so that it needs **at least 2× less memory**
  than with default dtypes (all values must stay the same).
* **2B** `load_annotation_small(path)` – load only what is needed from the annotation: a DataFrame with the columns
  `probe_id` and `symbol` (in this order). Drop control probes (`Sequence Type == "Control sequence"`) and probes
  without gene symbol; if several symbols are given (`"A /// B"`), keep the first. `symbol` must be `category`.
* **2C** `diet_means_chunked(long_path, annotation, chunksize)` – read the long file in chunks, merge every chunk with
  the annotation, and keep **sums and counts** per `(symbol, diet)`. Return the mean expression as a DataFrame
  with one row per gene symbol and one column per diet.
* **2D** `top_fastfood_vs_pellet(means, k)` – return the *k* genes with the largest
  log2 fold change `log2((FastFood + 1) / (Pellet + 1))` as a Series sorted in descending order.
* **Bonus** Time `diet_means_chunked` with chunk sizes 10,000, 100,000 and 1,000,000. What do you observe and why?

In [ ]:
def load_expression_small(path):
    # TODO 2A
    raise NotImplementedError


def load_annotation_small(path):
    # TODO 2B
    raise NotImplementedError


def diet_means_chunked(long_path, annotation, chunksize=100_000):
    # TODO 2C: for chunk in pd.read_csv(..., chunksize=chunksize): merge, groupby, agg(["sum", "count"])
    raise NotImplementedError


def top_fastfood_vs_pellet(means, k=10):
    # TODO 2D
    raise NotImplementedError

In [ ]:
check_exercise_2(load_expression_small, load_annotation_small, diet_means_chunked, top_fastfood_vs_pellet)

In [ ]:
# Bonus: time diet_means_chunked with different chunk sizes
# annotation = load_annotation_small(ANNOTATION)
# for size in (10_000, 100_000, 1_000_000):
#     with measure(f"chunksize = {size:,}"):
#         diet_means_chunked(LONG_FILE, annotation, chunksize=size)

# Beyond memory

## 3.1 A database on disk: SQLite
`sqlite3` is part of the Python standard library. A SQLite database is a single file; queries run **on disk**
and only the result comes back into memory. pandas writes to it with `to_sql` and reads with `read_sql_query`.

# Exercise 3 – SQL and choosing a strategy (10 min)
* **3A** Write a SQL query (string `QUERY_TOP5`) on `primates_diet.sqlite` that returns the **5 gene symbols with the
  highest mean expression in the HumanCafe diet**: two columns (symbol, mean value), highest first.
  *Hint:* `JOIN annotation a ON e.probe_id = a.probe_id`, `WHERE`, `GROUP BY`, `ORDER BY ... DESC`, `LIMIT`.
* **3B** For each scenario, decide on a strategy and justify it in one sentence (write your answers in the markdown cell):
  1. Count the reads in a 20 GB gzip-compressed FASTQ file (4 lines per read) on a laptop with 16 GB RAM.
  2. A 300 MB CSV file with expression values that your group analyses interactively every day.
  3. The mean expression per gene and condition in a 50 GB long-format table.
  4. The **median** expression per gene in the same 50 GB table.

In [ ]:
QUERY_TOP5 = None   # TODO 3A: replace None by your SQL query as a string

In [ ]:
check_exercise_3(QUERY_TOP5)
if QUERY_TOP5 is not None:
    with sqlite3.connect(DB_FILE) as con:
        print(pd.read_sql_query(QUERY_TOP5, con))

**3B – your answers:**
1. …
2. …
3. …
4. …

---
# Wrap-up (5 min)
1. **Measure first** – file size on disk ≠ memory in Python (`memory_usage(deep=True)`, `tracemalloc`).
2. **Stream** what you only need to see once: file iteration, generators, generator expressions.
3. **Load less** with pandas: `usecols`, small numeric dtypes, `category`.
4. **Chunk** large tables and combine **sums and counts**, never means of means.
5. **Store smartly** for repeated analyses: databases (SQLite) or columnar formats (Parquet).

In [ ]:
# Clean up the files created in this session (optional)
# LONG_FILE.unlink(missing_ok=True)
# DB_FILE.unlink(missing_ok=True)